7. Part D — Experiment 1: Inspect the Sparse Representation

In [1]:
import json
import gzip
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer

corpus = []

file_path = r'C:\Users\EGlaciers\Desktop\NLP-Apps\c4-train.00000-of-01024-30K.json.gz'

with gzip.open(file_path, 'rt', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line)
        corpus.append(data.get('text', ''))

vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(corpus)

#7.3. Kiểm tra kích thước
N = X.shape[0]
V = X.shape[1]
print(f"Number of documents (N) = {N}")
print(f"Vocabulary size (V) = {V}")
print(f"Matrix shape = {X.shape}")
print()

#7.4. Kiểm tra sparsity
nnz = X.nnz
sparsity = 1 - (nnz / (N * V))
print(f"Number of non-zero elements (nnz) = {nnz}")
print(f"Sparsity (S) = {sparsity:.6f}")
print()

feature_names = vectorizer.get_feature_names_out()

df = np.array((X > 0).sum(axis=0)).flatten()
top_df_indices = df.argsort()[-20:][::-1]

print("20 terms phổ biến nhất theo Document Frequency (DF):")
for idx in top_df_indices:
    print(f" - {feature_names[idx]}: DF = {df[idx]}")

idf = vectorizer.idf_
top_idf_indices = idf.argsort()[-20:][::-1]

print("\n20 terms có IDF cao nhất:")
for idx in top_idf_indices:
    print(f" - {feature_names[idx]}: IDF = {idf[idx]:.4f}")

doc_idx = 0
doc_vector = X[doc_idx].toarray().flatten()
top_tfidf_indices = doc_vector.argsort()[-20:][::-1]

print(f"\n20 terms có TF-IDF cao nhất trong Document {doc_idx}:")
for idx in top_tfidf_indices:
    if doc_vector[idx] > 0:
        print(f" - {feature_names[idx]}: TF-IDF = {doc_vector[idx]:.4f}")

Number of documents (N) = 30000
Vocabulary size (V) = 193540
Matrix shape = (30000, 193540)

Number of non-zero elements (nnz) = 4985822
Sparsity (S) = 0.999141

20 terms phổ biến nhất theo Document Frequency (DF):
 - the: DF = 27893
 - and: DF = 27423
 - to: DF = 26689
 - of: DF = 26031
 - in: DF = 25224
 - for: DF = 23651
 - is: DF = 22739
 - with: DF = 21405
 - on: DF = 20262
 - that: DF = 18370
 - this: DF = 17840
 - are: DF = 17594
 - it: DF = 17168
 - as: DF = 16467
 - at: DF = 16347
 - from: DF = 16316
 - be: DF = 16153
 - you: DF = 16094
 - by: DF = 15123
 - have: DF = 14852

20 terms có IDF cao nhất:
 - 00000: IDF = 10.6158
 - 00003: IDF = 10.6158
 - 000040: IDF = 10.6158
 - 00005: IDF = 10.6158
 - 0000856166: IDF = 10.6158
 - 0001042: IDF = 10.6158
 - 000116: IDF = 10.6158
 - 00012: IDF = 10.6158
 - 00015: IDF = 10.6158
 - 00016: IDF = 10.6158
 - 000165101: IDF = 10.6158
 - 0002: IDF = 10.6158
 - 00022: IDF = 10.6158
 - 000226: IDF = 10.6158
 - 000281: IDF = 10.6158
 - 확인하게: 

* Tại sao một document chỉ sử dụng một phần rất nhỏ vocabulary nhưng vector vẫn có
chiều (V)?

để ta có thể so sánh 2 văn bản bất kỳ, ví dụ như tính cosine similarity -> các vector văn bản phải nằm trong cùng một không gian vector

* So sánh 3 danh sách:
    - Danh sách 1: gồm các stop-words: a, an, the, and, ... phổ biến nhất trong văn bản
    - Danh sách 2: những từ hiếm xuất hiện trong văn bản
    - Danh sách 3: Những từ khóa quan trọng mang theo nội dung của văn bản

Một term xuất hiện rất nhiều trong corpus có nhất thiết có TF-IDF cao không?

không. vì IDF sẽ tiến dần về 0 -> TF-IDF nhỏ

Một term có IDF cao có nhất thiết có TF-IDF cao trong mọi document không?

không nhất thiết. IDF cao tức là từ đó hiếm gặp. nếu một document ko chứa từ đó thì DF = 0 -> TF-IDF = 0

Part F — Experiment 2: Preprocessing Ablation

In [6]:
import gzip
import json
import time
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import BertTokenizer

corpus = []

file_path = r'C:\Users\EGlaciers\Desktop\NLP-Apps\c4-train.00000-of-01024-30K.json.gz'

with gzip.open(file_path, 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        data = json.loads(line)
        corpus.append(data.get('text', ''))

        if i >= 29999:
            break


N = len(corpus)

print(f"Đã đọc {N:,} documents.")

queries = [
    "machine learning for healthcare",
    "covid-19 pandemic effects",
    "c++ programming tutorials"
]


pipeline_A = TfidfVectorizer(
    lowercase=True,
    token_pattern=r'(?u)\b\w+\b'
)

pipeline_B = TfidfVectorizer(
    lowercase=True,
    stop_words='english',
    token_pattern=r'(?u)\b[a-zA-Z]+\b'
)

bert_tokenizer = BertTokenizer.from_pretrained(
    'bert-base-uncased'
)

def bpe_tokenize(text):
    return bert_tokenizer.tokenize(text)


pipeline_C = TfidfVectorizer(
    tokenizer=bpe_tokenize,
    token_pattern=None,
    lowercase=False
)


pipelines = {
    "Pipeline A (Minimal)": pipeline_A,
    "Pipeline B (Normalized)": pipeline_B,
    "Pipeline C (Subword)": pipeline_C
}


def evaluate_pipeline(name, vectorizer):

    print("\n" + "=" * 60)
    print(f"Đang chạy: {name}")
    print("=" * 60)

    start_time = time.time()
    X = vectorizer.fit_transform(corpus)

    build_time = time.time() - start_time

    V = X.shape[1]

    nnz = X.nnz
    total_elements = N * V
    sparsity = 1.0 - (nnz / total_elements)

    analyzer = vectorizer.build_analyzer()

    total_tokens = 0

    for doc in corpus:
        tokens = analyzer(doc)
        total_tokens += len(tokens)

    avg_tokens = total_tokens / N

    vocab = vectorizer.vocabulary_
    total_query_tokens = 0
    oov_tokens = 0

    for query in queries:
        query_tokens = analyzer(query)

        total_query_tokens += len(query_tokens)

        for token in query_tokens:
            if token not in vocab:
                oov_tokens += 1


    if total_query_tokens > 0:
        oov_rate = (oov_tokens /total_query_tokens *100)

    else:
        oov_rate = 0

    q_vecs = vectorizer.transform(queries)

    sims = cosine_similarity(q_vecs,X)
    avg_matches = ((sims > 0).sum()/ len(queries))

    print(
        f"Vocabulary size     : {V:,}"
    )

    print(
        f"Avg tokens/doc      : {avg_tokens:.2f}"
    )

    print(
        f"Matrix sparsity     : "
        f"{sparsity * 100:.4f}%"
    )

    print(
        f"OOV rate (queries)  : "
        f"{oov_rate:.2f}%"
    )

    print(
        f"Avg matched docs/q  : "
        f"{avg_matches:.0f} tài liệu"
    )

    print(
        f"Execution time      : "
        f"{build_time:.2f} giây"
    )


# ============================================================
# 8. THỰC THI 3 PIPELINE
# ============================================================

for name, pipe in pipelines.items():

    evaluate_pipeline(
        name,
        pipe
    )

Đã đọc 30,000 documents.

Đang chạy: Pipeline A (Minimal)
Vocabulary size     : 193,837
Avg tokens/doc      : 369.70
Matrix sparsity     : 99.9122%
OOV rate (queries)  : 9.09%
Avg matched docs/q  : 9018 tài liệu
Execution time      : 15.15 giây

Đang chạy: Pipeline B (Normalized)


Token indices sequence length is longer than the specified maximum sequence length for this model (2531 > 512). Running this sequence through the model will result in indexing errors


Vocabulary size     : 167,107
Avg tokens/doc      : 188.36
Matrix sparsity     : 99.9286%
OOV rate (queries)  : 11.11%
Avg matched docs/q  : 1484 tài liệu
Execution time      : 8.19 giây

Đang chạy: Pipeline C (Subword)
Vocabulary size     : 28,339
Avg tokens/doc      : 465.10
Matrix sparsity     : 99.3200%
OOV rate (queries)  : 0.00%
Avg matched docs/q  : 15882 tài liệu
Execution time      : 46.33 giây


10. Part G — Application: Build a Document Search Engine

In [7]:
import gzip
import json
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

corpus = []
file_path = r'C:\Users\EGlaciers\Desktop\NLP-Apps\c4-train.00000-of-01024-30K.json.gz'

print(f"Loading 30K documents from {file_path}...")
with gzip.open(file_path, 'rt', encoding='utf-8') as f:
    for i, line in enumerate(f):
        data = json.loads(line)
        corpus.append(data.get('text', ''))
        if i >= 29999:  
            break

print("Building TF-IDF Index...")
vectorizer = TfidfVectorizer(lowercase=True, stop_words='english')
X_matrix = vectorizer.fit_transform(corpus)


def search_engine(query, vectorizer, tfidf_matrix, corpus, top_k=5):
    query_vec = vectorizer.transform([query])
    
    sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
    
    top_doc_ids = sim_scores.argsort()[-top_k:][::-1]
    
    print(f"\nQuery: '{query}'")
    print(f"{'Rank':<5} | {'Doc ID':<8} | {'Similarity':<10} | {'Document preview'}")
    print("-" * 85)
    
    for rank, doc_id in enumerate(top_doc_ids, start=1):
        score = sim_scores[doc_id]
        
        if score == 0.0:
            print(f"{rank:<5} | {'-':<8} | {score:<10.4f} | Không có tài liệu phù hợp.")
            continue
            
        preview = corpus[doc_id].replace('\n', ' ')[:60].strip() + "..."
        print(f"{rank:<5} | {doc_id:<8} | {score:<10.4f} | {preview}")


queries = [
    "medical image classification",
    "transformer language model",
    "deep learning healthcare",
    "natural language processing"
]

for q in queries:
    search_engine(q, vectorizer, X_matrix, corpus, top_k=5)

Loading 30K documents from C:\Users\EGlaciers\Desktop\NLP-Apps\c4-train.00000-of-01024-30K.json.gz...
Building TF-IDF Index...

Query: 'medical image classification'
Rank  | Doc ID   | Similarity | Document preview
-------------------------------------------------------------------------------------
1     | 18971    | 0.4240     | The new RTS Environmental Classification system (RTS GLT) is...
2     | 8527     | 0.3537     | History of maize classification. How races used in classific...
3     | 19908    | 0.2614     | Download League Of Legends Wallpapers in high-quality for yo...
4     | 15682    | 0.2505     | - Group Image: Provided functionality of group's image, user...
5     | 12658    | 0.2490     | This guidance is for pharmacists who handle, use and sell/su...

Query: 'transformer language model'
Rank  | Doc ID   | Similarity | Document preview
-------------------------------------------------------------------------------------
1     | 27936    | 0.4792     | hi, I am having

11. Part H — Evaluation

In [13]:
import csv
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

eval_vectorizer = TfidfVectorizer(lowercase=True, stop_words='english')
eval_matrix = eval_vectorizer.fit_transform(corpus)

ground_truth = {
    "medical image classification": [18971],
    "transformer language model": [27936, 25428],
    "deep learning healthcare": [6123, 11979, 11119],
    "natural language processing": [25428]
}

def calculate_precision_recall_at_k(retrieved_docs, relevant_docs, k=5):
    top_k_retrieved = retrieved_docs[:k]
    relevant_retrieved = set(top_k_retrieved).intersection(set(relevant_docs))
    num_relevant_retrieved = len(relevant_retrieved)
    
    precision_at_k = num_relevant_retrieved / k
    total_relevant = len(relevant_docs)
    recall_at_k = num_relevant_retrieved / total_relevant if total_relevant > 0 else 0.0
    
    return precision_at_k, recall_at_k

def calculate_rr(retrieved_docs, relevant_docs):
    for rank, doc_id in enumerate(retrieved_docs, start=1):
        if doc_id in relevant_docs:
            return 1.0 / rank
    return 0.0

def evaluate_search_engine(vectorizer, tfidf_matrix, ground_truth, k=5, output_file="results.csv"):
    precisions = []
    recalls = []
    reciprocal_ranks = []
    
    csv_data = []
    
    for query, relevant_docs in ground_truth.items():
        query_vec = vectorizer.transform([query])
        sim_scores = cosine_similarity(query_vec, tfidf_matrix).flatten()
        retrieved_docs = sim_scores.argsort()[-k:][::-1].tolist()
        
        p_at_k, r_at_k = calculate_precision_recall_at_k(retrieved_docs, relevant_docs, k)
        rr = calculate_rr(retrieved_docs, relevant_docs)
        
        precisions.append(p_at_k)
        recalls.append(r_at_k)
        reciprocal_ranks.append(rr)
        
        csv_data.append([query, f"{p_at_k:.2f}", f"{r_at_k:.2f}", f"{rr:.2f}"])
        
    mrr = np.mean(reciprocal_ranks)
    mean_p = np.mean(precisions)
    mean_r = np.mean(recalls)
    
    csv_data.append(["AVERAGE (MRR, Mean P@5, Mean R@5)", f"{mean_p:.2f}", f"{mean_r:.2f}", f"{mrr:.2f}"])
    
    with open(output_file, mode='w', encoding='utf-8', newline='') as file:
        writer = csv.writer(file)
        writer.writerow(['Query', 'P@5', 'R@5', 'RR']) 
        writer.writerows(csv_data)                     
        
    print(f"Đã lưu kết quả đánh giá vào file {output_file}")

evaluate_search_engine(eval_vectorizer, eval_matrix, ground_truth, k=5)

Đã lưu kết quả đánh giá vào file results.csv
